[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammad-zainal-muttaqin/NulisBuku/blob/main/website/notebooks/ch07.ipynb)

Notebook Bab 7 ini punya dua bagian. Bagian **Demo** tinggal Anda jalankan lalu amati keluarannya; bagian **Mini Project** berisi soal dan data yang Anda kerjakan sendiri.

Seleksi fitur membuang fitur redundan dan *noise*. Kita bandingkan metode *filter*, *wrapper*, dan *embedded*, semuanya di dalam *pipeline* agar tidak bocor.

## Persiapan

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

RANDOM_STATE = 42
print('Setup selesai.')

Setup selesai.


# Section 1 - Demo: Membandingkan Metode Seleksi Fitur

## Data: 50 fitur, hanya sebagian yang informatif

Delapan fitur informatif, lima redundan, sisanya *noise*.

In [2]:
X, y = make_classification(n_samples=2000, n_features=50, n_informative=8,
                           n_redundant=5, random_state=RANDOM_STATE)
print('Data:', X.shape)

Data: (2000, 50)


## Filter vs Wrapper vs Embedded

Setiap selektor berada di dalam *pipeline* sehingga di-*fit* ulang pada tiap *fold* latih (mencegah *selection leakage*).

In [3]:
def cv(steps):
    return cross_val_score(Pipeline(steps), X, y, cv=5).mean()

lr = LogisticRegression(max_iter=1000)
base = cv([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])
filt = cv([('sel', SelectKBest(mutual_info_classif, k=10)), ('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])
wrap = cv([('sc', StandardScaler()), ('sel', RFE(lr, n_features_to_select=10, step=5)), ('lr', LogisticRegression(max_iter=1000))])
emb = cv([('sel', SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))), ('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])

print(f'Semua 50 fitur (baseline) = {base:.3f}')
print(f'Filter  (MI, k=10)        = {filt:.3f}')
print(f'Wrapper (RFE, 10 fitur)   = {wrap:.3f}')
print(f'Embedded (tree importance)= {emb:.3f}')

Semua 50 fitur (baseline) = 0.724
Filter  (MI, k=10)        = 0.717
Wrapper (RFE, 10 fitur)   = 0.720
Embedded (tree importance)= 0.724


> 🔎 **Amati.** Ketiga metode mempertahankan akurasi mendekati baseline hanya dengan sekitar 10 dari 50 fitur, jadi model lebih ramping dan cepat. Bedanya pada biaya: *filter* paling murah (skor per fitur), *wrapper* (RFE) paling mahal karena melatih ulang model berkali-kali, *embedded* menyeleksi sekalian dari kepentingan fitur saat melatih. Karena selektor berada di dalam *pipeline*, pemilihan fitur tidak pernah mengintip data uji.

# Section 2 - Mini Project

## Soal

Anda diberi data berdimensi tinggi (80 fitur, banyak *noise*). Targetnya klasifikasi biner.

Tugas:

1. Bandingkan dua metode seleksi (misalnya *filter* dan *embedded*) di dalam *pipeline*.
2. Periksa **stabilitas** seleksi: fitur mana yang konsisten terpilih antar-*fold*?
3. Laporkan akurasi dan jumlah fitur terpilih tiap metode.

**Luaran:** kode perbandingan, daftar fitur stabil, dan 2-3 kalimat kesimpulan.

**Kriteria penilaian:** (a) selektor di dalam *pipeline*; (b) ada analisis stabilitas antar-*fold*; (c) perbandingan adil (model akhir sama).

In [4]:
# DATA AWAL (jangan diubah) - 80 fitur, 10 informatif, 8 redundan.
Xm, ym = make_classification(n_samples=1500, n_features=80, n_informative=10,
                             n_redundant=8, random_state=7)
print('Data:', Xm.shape)

Data: (1500, 80)


In [5]:
# Kerjakan di sini.
# Petunjuk: SelectKBest / SelectFromModel di dalam Pipeline; cek get_support() tiap fold.
